In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
sns.set()
%matplotlib inline

from sklearn.model_selection import train_test_split

In [ ]:
# Loading The Dataset
df = pd.read_csv('Reviews.csv')

In [ ]:
# Viewing The first 5 rows
df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
#Checking the dataset info
df.info()

In [ ]:
final_dataset = df[df.HelpfulnessDenominator <= df.HelpfulnessDenominator]

In [ ]:
# Checking Missing Values
df.isnull().sum()

In [ ]:
# EDA
print("Total number of reviews:", len(final_dataset))

In [ ]:
# Rating Distribution (Bar Chart)
plt.figure(figsize=(8,5))
sns.countplot(x=final_dataset['Score'], palette='viridis')
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()


In [ ]:
# Average Review Length by Rating
final_dataset['review_length'] = final_dataset['Text'].apply(lambda x: len(str(x).split()))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='Score', y='review_length', data=final_dataset, palette='magma')
plt.title("Average Review Length by Rating")
plt.xlabel("Rating")
plt.ylabel("Average Word Count")
plt.show()


In [ ]:
# Temporal Trends (Reviews Over Time)
final_dataset['Time'] = pd.to_datetime(final_dataset['Time'], unit='s')
final_dataset['year'] = final_dataset['Time'].dt.year


plt.figure(figsize=(10,5))
final_dataset['year'].value_counts().sort_index().plot(kind='line', marker='o')
plt.title("Reviews Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Reviews")
plt.grid(True)
plt.show()


In [ ]:
# Handling The Missing Values
df['ProfileName'] = df['ProfileName'].fillna(df['ProfileName'].mode()[0])
df['Summary'] = df['Summary'].fillna(df['Summary'].mode()[0])

In [ ]:
# Text Preprocessing
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [ ]:
from tqdm import tqdm
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
preprocessed_review = []

for sentence in tqdm(final_dataset['Text'].values):
    sentence = re.sub(r"http\S+", "", sentence)
    sentence = BeautifulSoup(sentence, 'lxml').get_text()
    sentence = re.sub('[^a-zA-Z]+', ' ', sentence)
    sentence = ' '.join(
        e.lower() for e in sentence.split()
        if e.lower() not in stop_words
    )
    preprocessed_review.append(sentence.strip())


In [ ]:
preprocessed_review

In [ ]:
!pip install ydata-profiling

In [ ]:
# EDA
from ydata_profiling import ProfileReport
reports = ProfileReport(final_dataset)
reports.to_file(output_file='output.html')

In [ ]:
# Glove
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
glove_path = "/content/glove.6B.300d.txt"

In [ ]:
def load_glove(glove_filepath, dim=300):
    import numpy as np
    from tqdm import tqdm

    glove = {}
    with open(glove_filepath, 'r', encoding='utf8', errors='ignore') as f:
        for line in tqdm(f, desc="Loading GloVe"):
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            if len(vector) == dim:
                glove[word] = vector
    return glove

glove = load_glove(glove_path, dim=300)
print("GloVe words loaded:", len(glove))

In [ ]:
import pandas as pd

# Assuming your dataset has a 'Score' column
processed_data = pd.DataFrame({
    'Text': preprocessed_review,
    'Score': final_dataset['Score'].values
})

# Now you can safely do train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    processed_data['Text'], processed_data['Score'], test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
import numpy as np

embedding_index = {}
with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_dim = 100
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

num_classes = 5  # Ratings 1-5

model = Sequential()
model.add(Embedding(input_dim=num_words,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))  # Freeze GloVe
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(
    X_train_pad, y_train - 1,  # subtract 1 for sparse_categorical_crossentropy (labels 0-4)
    validation_split=0.1,
    epochs=10,
    batch_size=128
)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import numpy as np

y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1) + 1  # add 1 to convert back to 1-5 ratings
y_true = y_test.values

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))


In [ ]:

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[1,2,3,4,5],
            yticklabels=[1,2,3,4,5])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for Rating Prediction")
plt.show()